In [7]:
# ============================================================
# CELL 1: CÀI ĐẶT VÀ IMPORT THƯ VIỆN
# ============================================================

import wfdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

In [8]:
# ============================================================
# CELL 2: DANH SÁCH RECORD MIT-BIH
# ============================================================

db_name = "mitdb"

records = [
    "100", "101", "102", "103", "104", "105", "106", "107", "108", "109",
    "111", "112", "113", "114", "115", "116", "117", "118", "119",
    "121", "122", "123", "124",
    "200", "201", "202", "203", "205", "207", "208", "209",
    "210", "212", "213", "214", "215", "217", "219", "220",
    "221", "222", "223", "228", "230", "231", "232", "233", "234"
]

records_1xx = [r for r in records if r.startswith("1")]
records_2xx = [r for r in records if r.startswith("2")]

print("Total records:", len(records))
print("1xx records:", len(records_1xx))
print("2xx records:", len(records_2xx))

Total records: 48
1xx records: 23
2xx records: 25


In [ ]:
# ============================================================
# CELL 3: TẠO BẢNG THÔNG TIN RECORD
# ============================================================

record_name = "100"

header = wfdb.rdheader(record_name, pn_dir=db_name)

info_df = pd.DataFrame({
    "Field": [
        "Record name",
        "Number of leads",
        "Sampling rate (Hz)",
        "Number of samples",
        "Lead names"
    ],
    "Value": [
        header.record_name,
        header.n_sig,
        int(header.fs),
        header.sig_len,
        ", ".join(header.sig_name)
    ]
})

info_df

,Field,Value
0,Record name,100
1,Number of leads,2
2,Sampling rate (Hz),360
3,Number of samples,650000
4,Lead names,"MLII, V5"


In [18]:
# ============================================================
# CELL 4: BẢNG LEAD CỦA CÁC RECORD 1xx
# ============================================================

lead_rows_1xx = []

for rec in records_1xx:
    h = wfdb.rdheader(rec, pn_dir=db_name)

    lead_rows.append({
        "Record (1xx)": rec,
        "Channel 1 Lead": h.sig_name[0],
        "Channel 2 Lead": h.sig_name[1]
    })

lead_1xx_df = pd.DataFrame(lead_rows)

lead_1xx_df



,Record (1xx),Channel 1 Lead,Channel 2 Lead
0,100,MLII,V5
1,101,MLII,V1
2,102,V5,V2
3,103,MLII,V2
4,104,V5,V2
5,105,MLII,V1
6,106,MLII,V1
7,107,MLII,V1
8,108,MLII,V1
9,109,MLII,V1


In [17]:
# ============================================================
# CELL 4: BẢNG LEAD CỦA CÁC RECORD 2xx
# ============================================================

lead_rows_2xx = []

for rec in records_2xx:
    h = wfdb.rdheader(rec, pn_dir=db_name)

    lead_rows_2xx.append({
        "Record (2xx)": rec,
        "Channel 1 Lead": h.sig_name[0],
        "Channel 2 Lead": h.sig_name[1]
    })

lead_2xx_df = pd.DataFrame(lead_rows_2xx)

lead_2xx_df

,Record (2xx),Channel 1 Lead,Channel 2 Lead
0,200,MLII,V1
1,201,MLII,V1
2,202,MLII,V1
3,203,MLII,V1
4,205,MLII,V1
5,207,MLII,V1
6,208,MLII,V1
7,209,MLII,V1
8,210,MLII,V1
9,212,MLII,V1


In [ ]:
# ============================================================
# CELL 5: THỐNG KÊ SỐ RECORD CÓ TỪNG LEAD
# ============================================================

all_leads_per_record = []

for rec in records:
    h = wfdb.rdheader(rec, pn_dir=db_name)

    unique_leads = set(h.sig_name)

    for lead in unique_leads:
        all_leads_per_record.append(lead)

lead_counter = Counter(all_leads_per_record)

lead_summary_df = pd.DataFrame({
    "Lead": list(lead_counter.keys()),
    "Record Count": list(lead_counter.values())
})

lead_summary_df["Percentage (%)"] = (
    lead_summary_df["Record Count"] / len(records) * 100
).round(2)

lead_summary_df = lead_summary_df.sort_values(
    by="Record Count",
    ascending=False
).reset_index(drop=True)

lead_summary_df

,Lead,Record Count,Percentage (%)
0,MLII,46,95.83
1,V1,40,83.33
2,V5,5,10.42
3,V2,4,8.33
4,V4,1,2.08


In [13]:
# ============================================================
# CELL 7: THỐNG KÊ LABEL ANNOTATION 1xx VÀ 2xx
# ============================================================

def count_labels(record_list):
    """
    Đọc file .atr của từng record và đếm số lượng từng annotation symbol.
    """
    label_counter = Counter()

    for rec in record_list:
        ann = wfdb.rdann(rec, "atr", pn_dir=db_name)
        label_counter.update(ann.symbol)

    return label_counter


label_count_1xx = count_labels(records_1xx)
label_count_2xx = count_labels(records_2xx)

total_1xx = sum(label_count_1xx.values())
total_2xx = sum(label_count_2xx.values())

all_labels = sorted(
    set(label_count_1xx.keys()) | set(label_count_2xx.keys()),
    key=lambda x: label_count_1xx[x] + label_count_2xx[x],
    reverse=True
)

label_rows = []

for label in all_labels:
    count_1xx = label_count_1xx[label]
    count_2xx = label_count_2xx[label]
    total_count = count_1xx + count_2xx

    label_rows.append({
        "Label": label,
        "1xx Count": count_1xx,
        "1xx (%)": round(count_1xx / total_1xx * 100, 4),
        "2xx Count": count_2xx,
        "2xx (%)": round(count_2xx / total_2xx * 100, 4),
        "Total Count": total_count,
        "Total (%)": round(total_count / (total_1xx + total_2xx) * 100, 4)
    })

label_distribution_df = pd.DataFrame(label_rows)

label_distribution_df

,Label,1xx Count,1xx (%),2xx Count,2xx (%),Total Count,Total (%)
0,N,31545,65.4135,43507,67.5333,75052,66.6258
1,L,4615,9.5699,3460,5.3708,8075,7.1684
2,R,3697,7.6663,3562,5.5291,7259,6.4440
3,V,1346,2.7911,5784,8.9782,7130,6.3295
4,/,5486,11.3761,1542,2.3936,7028,6.2390
5,A,155,0.3214,2391,3.7114,2546,2.2602
6,+,227,0.4707,1064,1.6516,1291,1.1461
7,f,722,1.4972,260,0.4036,982,0.8717
8,F,13,0.0270,790,1.2263,803,0.7128
9,~,278,0.5765,338,0.5247,616,0.5468
